In [2]:
import pandas as pd
import numpy as np

pokemon_df = pd.read_csv('data/pokemon.csv', sep="\t", engine="python", encoding="utf-8")
print(pokemon_df.head())

favourites_df = pd.read_csv('data/favourites.csv', sep="\t", engine="python", encoding="utf-8")
print(favourites_df.head())

# Merge using english_name and Pokémon columns
pokemon_df = pokemon_df.merge(
    favourites_df,
    left_on='english_name',
    right_on='Pokémon',
    how='left'
)
pokemon_df = pokemon_df.drop(columns=['Pokémon'])
pokemon_df = pokemon_df.rename(columns={
    'Number of #1 Votes': 'votes_first',
    'Number of Top 6 Votes': 'votes_top_6'
})
print(pokemon_df.columns)

   national_number gen english_name japanese_name primary_type secondary_type  \
0                1   I    Bulbasaur   Fushigidane        grass         poison   
1                2   I      Ivysaur    Fushigisou        grass         poison   
2                3   I     Venusaur   Fushigibana        grass         poison   
3                4   I   Charmander      Hitokage         fire            NaN   
4                5   I   Charmeleon       Lizardo         fire            NaN   

   classification percent_male percent_female  height_m  ...  evochain_1  \
0    Seed Pokémon        88.14          11.86       0.7  ...      Level    
1    Seed Pokémon        88.14          11.86       1.0  ...      Level    
2    Seed Pokémon        88.14          11.86       2.0  ...      Level    
3  Lizard Pokémon        88.14          11.86       0.6  ...      Level    
4   Flame Pokémon        88.14          11.86       1.1  ...      Level    

   evochain_2  evochain_3  evochain_4  evochain_5  evoch

In [3]:
print(pokemon_df.columns)
pokemon_df['num_abilities'] = pokemon_df[['abilities_0', 'abilities_1', 'abilities_2', 'abilities_hidden']].notnull().sum(axis=1)
pokemon_df = pokemon_df.drop(columns=[
    'national_number', 'japanese_name', 'description',
    'abilities_0', 'abilities_1', 'abilities_2', 'abilities_hidden',
])
print(pokemon_df.columns)

Index(['national_number', 'gen', 'english_name', 'japanese_name',
       'primary_type', 'secondary_type', 'classification', 'percent_male',
       'percent_female', 'height_m', 'weight_kg', 'capture_rate',
       'base_egg_steps', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense',
       'speed', 'abilities_0', 'abilities_1', 'abilities_2',
       'abilities_hidden', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying',
       'against_psychict', 'against_bug', 'against_rock', 'against_ghost',
       'against_dragon', 'against_dark', 'against_steel', 'against_fairy',
       'is_sublegendary', 'is_legendary', 'is_mythical', 'evochain_0',
       'evochain_1', 'evochain_2', 'evochain_3', 'evochain_4', 'evochain_5',
       'evochain_6', 'gigantamax', 'mega_evolution', 'mega_evolution_alt',
       'description', 'votes_first', 'votes_top_6'],
      dty

In [4]:
evochain_cols = ['evochain_0', 'evochain_1', 'evochain_2', 'evochain_3', 'evochain_4', 'evochain_5', 'evochain_6']
pokemon_df['evo_length'] = pokemon_df[evochain_cols].notna().sum(axis=1)
pokemon_df = pokemon_df.drop(columns=evochain_cols)

In [5]:
unique_classes =pokemon_df["classification"].unique()
print("Number of unique classes:", len(unique_classes))
unique_primary_types = pokemon_df["primary_type"].unique()
print("Number of unique primary types:", len(unique_primary_types))
unique_secondary_types = pokemon_df["secondary_type"].unique()
print("Number of unique secondary types:", len(unique_secondary_types))

pokemon_df = pokemon_df.drop(columns=['classification'])

Number of unique classes: 647
Number of unique primary types: 18
Number of unique secondary types: 19


In [6]:
pokemon_df['percent_male'] = pokemon_df['percent_male'].fillna('0')
pokemon_df['percent_female'] = pokemon_df['percent_female'].fillna('0')
pokemon_df.loc[pokemon_df['percent_male'].str.match(r'50\*'), 'percent_male'] = '50'
pokemon_df.loc[pokemon_df['percent_female'].str.match(r'50\*'), 'percent_female'] = '50'

print("Unique values in percent_male:", pokemon_df['percent_male'].unique())


Unique values in percent_male: ['88.14' '50.2' '50' '0' '100' '24.9' '75.49' '88.1' '24.6' '75.4' '11.2']


In [7]:
pokemon_df['has_mega_evolution'] = (
    pokemon_df['mega_evolution'].notna() | pokemon_df['mega_evolution_alt'].notna()
).astype(int)

pokemon_df['has_gigantamax'] = pokemon_df['gigantamax'].notna().astype(int)

pokemon_df = pokemon_df.drop(columns=['mega_evolution', 'mega_evolution_alt', 'gigantamax'])

In [8]:
pokemon_df['rarity'] = (
    1 * pokemon_df['is_sublegendary'] +
    2 * pokemon_df['is_legendary'] +
    3 * pokemon_df['is_mythical']
)
assert pokemon_df['rarity'].max() == 3, "Rarity encoding error"
pokemon_df = pokemon_df.drop(columns=['is_sublegendary', 'is_legendary', 'is_mythical'])
print(pokemon_df.columns)

Index(['gen', 'english_name', 'primary_type', 'secondary_type', 'percent_male',
       'percent_female', 'height_m', 'weight_kg', 'capture_rate',
       'base_egg_steps', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense',
       'speed', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying',
       'against_psychict', 'against_bug', 'against_rock', 'against_ghost',
       'against_dragon', 'against_dark', 'against_steel', 'against_fairy',
       'votes_first', 'votes_top_6', 'num_abilities', 'evo_length',
       'has_mega_evolution', 'has_gigantamax', 'rarity'],
      dtype='object')


In [9]:
type_columns = ['primary_type', 'secondary_type']
string_columns = type_columns + ['english_name']
gen_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8}
pokemon_df['gen'] = pokemon_df['gen'].map(gen_map)

columns_to_cast = [col for col in pokemon_df.columns if col not in string_columns]

pokemon_df[columns_to_cast] = pokemon_df[columns_to_cast].astype(float)

In [12]:
print("Primary Types:", unique_primary_types)
print("Secondary Types:", unique_secondary_types)
pokemon_df['secondary_type'] = pokemon_df['secondary_type'].fillna('No secondary type')
unique_secondary_types_clean = pokemon_df['secondary_type'].unique()
print("Secondary Types (clean):", unique_secondary_types_clean)

# encode primary_type as integer class labels and replace the column with class_primary
labels, uniques = pd.factorize(pokemon_df['primary_type'])
pokemon_df['class_primary'] = labels.astype(int)
mapping = {v: i for i, v in enumerate(uniques)}
pokemon_df.drop(columns=['primary_type'], inplace=True)

print("class_primary mapping:", mapping)

labels_sec, uniques_sec = pd.factorize(pokemon_df['secondary_type'])
pokemon_df['class_secondary'] = labels_sec.astype(int)
mapping_sec = {v: i for i, v in enumerate(uniques_sec)}
pokemon_df.drop(columns=['secondary_type'], inplace=True)

print("class_secondary mapping:", mapping_sec)

Primary Types: ['grass' 'fire' 'water' 'bug' 'normal' 'poison' 'electric' 'ground'
 'fairy' 'fighting' 'psychic' 'rock' 'ghost' 'ice' 'dragon' 'dark' 'steel'
 'flying']
Secondary Types: ['poison' nan 'flying' 'dark' 'electric' 'ice' 'ground' 'fairy' 'grass'
 'fighting' 'psychic' 'steel' 'fire' 'rock' 'water' 'dragon' 'ghost' 'bug'
 'normal']
Secondary Types (clean): ['poison' 'No secondary type' 'flying' 'dark' 'electric' 'ice' 'ground'
 'fairy' 'grass' 'fighting' 'psychic' 'steel' 'fire' 'rock' 'water'
 'dragon' 'ghost' 'bug' 'normal']
class_primary mapping: {'grass': 0, 'fire': 1, 'water': 2, 'bug': 3, 'normal': 4, 'poison': 5, 'electric': 6, 'ground': 7, 'fairy': 8, 'fighting': 9, 'psychic': 10, 'rock': 11, 'ghost': 12, 'ice': 13, 'dragon': 14, 'dark': 15, 'steel': 16, 'flying': 17}
class_secondary mapping: {'poison': 0, 'No secondary type': 1, 'flying': 2, 'dark': 3, 'electric': 4, 'ice': 5, 'ground': 6, 'fairy': 7, 'grass': 8, 'fighting': 9, 'psychic': 10, 'steel': 11, 'fire': 12,

In [13]:
for col in pokemon_df.columns:
    unique_vals = pokemon_df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

Column: gen
Unique values (8): [1. 2. 3. 4. 5. 6. 7. 8.]

Column: english_name
Unique values (898): ['Bulbasaur' 'Ivysaur' 'Venusaur' 'Charmander' 'Charmeleon' 'Charizard'
 'Squirtle' 'Wartortle' 'Blastoise' 'Caterpie' 'Metapod' 'Butterfree'
 'Weedle' 'Kakuna' 'Beedrill' 'Pidgey' 'Pidgeotto' 'Pidgeot' 'Rattata'
 'Raticate' 'Spearow' 'Fearow' 'Ekans' 'Arbok' 'Pikachu' 'Raichu'
 'Sandshrew' 'Sandslash' 'Nidoran♀' 'Nidorina' 'Nidoqueen' 'Nidoran♂'
 'Nidorino' 'Nidoking' 'Clefairy' 'Clefable' 'Vulpix' 'Ninetales'
 'Jigglypuff' 'Wigglytuff' 'Zubat' 'Golbat' 'Oddish' 'Gloom' 'Vileplume'
 'Paras' 'Parasect' 'Venonat' 'Venomoth' 'Diglett' 'Dugtrio' 'Meowth'
 'Persian' 'Psyduck' 'Golduck' 'Mankey' 'Primeape' 'Growlithe' 'Arcanine'
 'Poliwag' 'Poliwhirl' 'Poliwrath' 'Abra' 'Kadabra' 'Alakazam' 'Machop'
 'Machoke' 'Machamp' 'Bellsprout' 'Weepinbell' 'Victreebel' 'Tentacool'
 'Tentacruel' 'Geodude' 'Graveler' 'Golem' 'Ponyta' 'Rapidash' 'Slowpoke'
 'Slowbro' 'Magnemite' 'Magneton' "Farfetch'd" 'Do

In [14]:
pokemon_df.to_csv('data/data_clean.csv', sep=',', index=False)